# Mistral-Small-3.2-24B-Instruct-2506 — Structured JSON Benchmarks

This notebook tests **Mistral-Small-3.2-24B-Instruct-2506** on structured output tasks:
- JSON generation
- Schema adherence
- Function calling format
- Key-value extraction from unstructured text

Model: [mistralai/Mistral-Small-3.2-24B-Instruct-2506](https://huggingface.co/mistralai/Mistral-Small-3.2-24B-Instruct-2506)

In [ ]:
%pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124
%pip install "transformers>=4.51,<5" accelerate sentencepiece
%pip install mistral-common>=1.6.2 --upgrade
%pip install jsonschema

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

In [ ]:
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version:    {torch.version.cuda}')
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    print(f'VRAM:            {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected! Check CUDA/driver compatibility.')

In [ ]:
from datetime import datetime, timedelta
import torch
import json
import re
import time

from mistral_common.protocol.instruct.request import ChatCompletionRequest
from mistral_common.tokens.tokenizers.mistral import MistralTokenizer
from huggingface_hub import hf_hub_download
from transformers import Mistral3ForConditionalGeneration

In [ ]:
model_name = "mistralai/Mistral-Small-3.2-24B-Instruct-2506"

In [ ]:
def load_system_prompt(repo_id: str, filename: str) -> str:
    file_path = hf_hub_download(repo_id=repo_id, filename=filename)
    with open(file_path, "r") as f:
        system_prompt = f.read()
    today = datetime.today().strftime("%Y-%m-%d")
    yesterday = (datetime.today() - timedelta(days=1)).strftime("%Y-%m-%d")
    model_name_local = repo_id.split("/")[-1]
    return system_prompt.format(name=model_name_local, today=today, yesterday=yesterday)

SYSTEM_PROMPT = load_system_prompt(model_name, "SYSTEM_PROMPT.txt")
print("System prompt loaded.")

In [ ]:
tokenizer = MistralTokenizer.from_hf_hub(model_name)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# Verify device
device = next(model.parameters()).device
print(f'Model loaded on: {device}')
if device.type == 'cpu':
    print('WARNING: Model is on CPU! GPU is not being used.')
else:
    print(f'OK: Model is on GPU ({torch.cuda.get_device_name(device)})')

## Quick Test — Simple JSON Generation

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are helpful assistant. Respond with ONLY valid JSON. No markdown, no code blocks, no explanation."
    },
    {
        "role": "user",
        "content": "Generate exactly one JSON object for a person with name, age, email, and city. Use realistic values."
    }
]

In [ ]:
tokenized = tokenizer.encode_chat_completion(ChatCompletionRequest(messages=messages))
input_ids = torch.tensor([tokenized.tokens])
attention_mask = torch.ones_like(input_ids)

In [ ]:
outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.15,
)

In [ ]:
decoded_output = tokenizer.decode(outputs[0][len(tokenized.tokens):])
print(decoded_output)

### Extraction Test — Harder Task

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant. Respond with ONLY valid JSON. No markdown, no code blocks, no explanation."
    },
    {
        "role": "user",
        "content": """Extract contact information from the following text into a JSON object with fields: name, phone, email, company, title.
        
Text: "John Smith is a Senior Software Engineer at TechCorp Inc. You can reach him at john.smith@techcorp.com or call (555) 123-4567."
        """
    }
]

In [ ]:
tokenized = tokenizer.encode_chat_completion(ChatCompletionRequest(messages=messages))
input_ids = torch.tensor([tokenized.tokens])
attention_mask = torch.ones_like(input_ids)

outputs = model.generate(
    input_ids=input_ids,
    attention_mask=attention_mask,
    max_new_tokens=512,
    do_sample=True,
    temperature=0.15,
)

response = tokenizer.decode(outputs[0][len(tokenized.tokens):])
print(response)

---
## Full Benchmark Suite

In [ ]:
SIMPLE_JSON_TASKS = [
    {
        "id": "json_simple_person",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Generate a JSON object for a person with the following fields: name (string), age (number), email (string), and city (string). Use realistic values.",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "email": {"type": "string"},
                "city": {"type": "string"}
            },
            "required": ["name", "age", "email", "city"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_simple_product",
        "category": "json_generation",
        "difficulty": "easy",
        "prompt": "Create a JSON object for a product listing with: product_name (string), price (number), in_stock (boolean), and category (string).",
        "schema": {
            "type": "object",
            "properties": {
                "product_name": {"type": "string"},
                "price": {"type": "number"},
                "in_stock": {"type": "boolean"},
                "category": {"type": "string"}
            },
            "required": ["product_name", "price", "in_stock", "category"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_nested_address",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Generate a JSON object for a user profile. It must have: name (string), age (number), and address (object with street, city, state, zip).",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "age": {"type": "number"},
                "address": {
                    "type": "object",
                    "properties": {
                        "street": {"type": "string"},
                        "city": {"type": "string"},
                        "state": {"type": "string"},
                        "zip": {"type": "string"}
                    },
                    "required": ["street", "city", "state", "zip"],
                    "additionalProperties": False
                }
            },
            "required": ["name", "age", "address"],
            "additionalProperties": False
        }
    },
    {
        "id": "json_array_orders",
        "category": "json_generation",
        "difficulty": "medium",
        "prompt": "Create a JSON array containing 3 order objects. Each order should have: order_id (string), items (array of strings), total (number), and status (string that must be one of: pending, shipped, delivered).",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "order_id": {"type": "string"},
                    "items": {
                        "type": "array",
                        "items": {"type": "string"}
                    },
                    "total": {"type": "number"},
                    "status": {"type": "string", "enum": ["pending", "shipped", "delivered"]}
                },
                "required": ["order_id", "items", "total", "status"],
                "additionalProperties": False
            },
            "minItems": 3,
            "maxItems": 3
        }
    },
    {
        "id": "json_complex_api",
        "category": "json_generation",
        "difficulty": "hard",
        "prompt": "Generate a JSON object representing an API response. It should have: status (number), message (string), data (object with users array, where each user has id, name, email, role where role is one of admin/user/moderator), and metadata (object with total_count, page, per_page).",
        "schema": {
            "type": "object",
            "properties": {
                "status": {"type": "number"},
                "message": {"type": "string"},
                "data": {
                    "type": "object",
                    "properties": {
                        "users": {
                            "type": "array",
                            "items": {
                                "type": "object",
                                "properties": {
                                    "id": {"type": "number"},
                                    "name": {"type": "string"},
                                    "email": {"type": "string"},
                                    "role": {"type": "string", "enum": ["admin", "user", "moderator"]}
                                },
                                "required": ["id", "name", "email", "role"],
                                "additionalProperties": False
                            }
                        }
                    },
                    "required": ["users"],
                    "additionalProperties": False
                },
                "metadata": {
                    "type": "object",
                    "properties": {
                        "total_count": {"type": "number"},
                        "page": {"type": "number"},
                        "per_page": {"type": "number"}
                    },
                    "required": ["total_count", "page", "per_page"],
                    "additionalProperties": False
                }
            },
            "required": ["status", "message", "data", "metadata"],
            "additionalProperties": False
        }
    }
]

SCHEMA_ADHERENCE_TASKS = [
    {
        "id": "schema_weather",
        "category": "schema_adherence",
        "difficulty": "easy",
        "prompt": "Output a valid JSON object matching this exact schema. Generate realistic weather data:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"location\":{\"type\":\"string\"},\"temperature\":{\"type\":\"number\"},\"unit\":{\"type\":\"string\",\"enum\":[\"celsius\",\"fahrenheit\"]},\"conditions\":{\"type\":\"string\"},\"humidity\":{\"type\":\"number\",\"minimum\":0,\"maximum\":100}},\"required\":[\"location\",\"temperature\",\"unit\",\"conditions\",\"humidity\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string"},
                "temperature": {"type": "number"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                "conditions": {"type": "string"},
                "humidity": {"type": "number", "minimum": 0, "maximum": 100}
            },
            "required": ["location", "temperature", "unit", "conditions", "humidity"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_database_record",
        "category": "schema_adherence",
        "difficulty": "medium",
        "prompt": "Generate a JSON object matching this schema representing a database record. Fill in realistic values:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"id\":{\"type\":\"string\",\"pattern\":\"^[a-f0-9]{8}$\"},\"created_at\":{\"type\":\"string\",\"format\":\"date-time\"},\"type\":{\"type\":\"string\",\"enum\":[\"customer\",\"vendor\",\"employee\"]},\"active\":{\"type\":\"boolean\"},\"tags\":{\"type\":\"array\",\"items\":{\"type\":\"string\"},\"maxItems\":5}},\"required\":[\"id\",\"created_at\",\"type\",\"active\",\"tags\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "id": {"type": "string", "pattern": "^[a-f0-9]{8}$"},
                "created_at": {"type": "string", "format": "date-time"},
                "type": {"type": "string", "enum": ["customer", "vendor", "employee"]},
                "active": {"type": "boolean"},
                "tags": {
                    "type": "array",
                    "items": {"type": "string"},
                    "maxItems": 5
                }
            },
            "required": ["id", "created_at", "type", "active", "tags"],
            "additionalProperties": False
        }
    },
    {
        "id": "schema_config_file",
        "category": "schema_adherence",
        "difficulty": "hard",
        "prompt": "Generate a valid JSON configuration object matching this schema for a web server config:\n\nSchema:\n{\"type\":\"object\",\"properties\":{\"server\":{\"type\":\"object\",\"properties\":{\"host\":{\"type\":\"string\"},\"port\":{\"type\":\"integer\",\"minimum\":1,\"maximum\":65535},\"ssl\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"cert_path\":{\"type\":\"string\"},\"key_path\":{\"type\":\"string\"}},\"required\":[\"enabled\"]}},\"required\":[\"host\",\"port\",\"ssl\"]},\"logging\":{\"type\":\"object\",\"properties\":{\"level\":{\"type\":\"string\",\"enum\":[\"debug\",\"info\",\"warn\",\"error\"]},\"file\":{\"type\":\"string\"},\"rotate\":{\"type\":\"boolean\"}},\"required\":[\"level\"]},\"cors\":{\"type\":\"object\",\"properties\":{\"enabled\":{\"type\":\"boolean\"},\"origins\":{\"type\":\"array\",\"items\":{\"type\":\"string\"}},\"methods\":{\"type\":\"array\",\"items\":{\"type\":\"string\",\"enum\":[\"GET\",\"POST\",\"PUT\",\"DELETE\",\"PATCH\"]}}},\"required\":[\"enabled\"]}},\"required\":[\"server\",\"logging\",\"cors\"],\"additionalProperties\":false}",
        "schema": {
            "type": "object",
            "properties": {
                "server": {
                    "type": "object",
                    "properties": {
                        "host": {"type": "string"},
                        "port": {"type": "integer"},
                        "ssl": {
                            "type": "object",
                            "properties": {
                                "enabled": {"type": "boolean"},
                                "cert_path": {"type": "string"},
                                "key_path": {"type": "string"}
                            },
                            "required": ["enabled"],
                            "additionalProperties": False
                        }
                    },
                    "required": ["host", "port", "ssl"],
                    "additionalProperties": False
                },
                "logging": {
                    "type": "object",
                    "properties": {
                        "level": {"type": "string", "enum": ["debug", "info", "warn", "error"]},
                        "file": {"type": "string"},
                        "rotate": {"type": "boolean"}
                    },
                    "required": ["level"],
                    "additionalProperties": False
                },
                "cors": {
                    "type": "object",
                    "properties": {
                        "enabled": {"type": "boolean"},
                        "origins": {"type": "array", "items": {"type": "string"}},
                        "methods": {
                            "type": "array",
                            "items": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]}
                        }
                    },
                    "required": ["enabled"],
                    "additionalProperties": False
                }
            },
            "required": ["server", "logging", "cors"],
            "additionalProperties": False
        }
    }
]

FUNCTION_CALLING_TASKS = [
    {
        "id": "funcall_get_weather",
        "category": "function_calling",
        "difficulty": "easy",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"get_weather\", \"description\": \"Get current weather for a location\", \"parameters\": {\"type\": \"object\", \"properties\": {\"location\": {\"type\": \"string\", \"description\": \"City name\"}, \"unit\": {\"type\": \"string\", \"enum\": [\"celsius\", \"fahrenheit\"]}}, \"required\": [\"location\"]}}\n\nThe user asks: \"What's the weather like in San Francisco?\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "get_weather",
        "expected_params_keys": ["location"]
    },
    {
        "id": "funcall_search_multi",
        "category": "function_calling",
        "difficulty": "medium",
        "prompt": "You are a helpful assistant with access to the following functions:\n\n1. {\"name\": \"search_web\", \"description\": \"Search the web for information\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\"}, \"num_results\": {\"type\": \"integer\", \"default\": 10}}, \"required\": [\"query\"]}}\n\n2. {\"name\": \"send_email\", \"description\": \"Send an email\", \"parameters\": {\"type\": \"object\", \"properties\": {\"to\": {\"type\": \"string\"}, \"subject\": {\"type\": \"string\"}, \"body\": {\"type\": \"string\"}}, \"required\": [\"to\", \"subject\", \"body\"]}}\n\nThe user asks: \"Search for the best restaurants in NYC and email the results to john@example.com\"\n\nRespond with the appropriate function call(s) in JSON format. If multiple calls are needed, use an array. Use the format: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "arguments": {"type": "object", "additionalProperties": True}
                },
                "required": ["name", "arguments"],
                "additionalProperties": False
            },
            "minItems": 1,
            "maxItems": 2
        },
        "expected_functions": ["search_web", "send_email"],
        "expected_params_keys": [["query", "num_results"], ["to", "subject", "body"]]
    },
    {
        "id": "funcall_database_query",
        "category": "function_calling",
        "difficulty": "hard",
        "prompt": "You are a helpful assistant with access to the following function:\n\n{\"name\": \"query_database\", \"description\": \"Execute a SQL query on the database\", \"parameters\": {\"type\": \"object\", \"properties\": {\"query\": {\"type\": \"string\", \"description\": \"SQL query to execute\"}, \"database\": {\"type\": \"string\", \"enum\": [\"production\", \"staging\", \"analytics\"]}, \"limit\": {\"type\": \"integer\", \"default\": 100, \"maximum\": 1000}, \"format\": {\"type\": \"string\", \"enum\": [\"json\", \"csv\"], \"default\": \"json\"}}, \"required\": [\"query\", \"database\"]}}\n\nThe user asks: \"Get me the top 50 customers by revenue from the production database in CSV format\"\n\nRespond with a function call in JSON format using: {\"name\": \"...\", \"arguments\": {...}}",
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "arguments": {"type": "object", "additionalProperties": True}
            },
            "required": ["name", "arguments"],
            "additionalProperties": False
        },
        "expected_function": "query_database",
        "expected_params_keys": ["query", "database", "limit", "format"]
    }
]

EXTRACTION_TASKS = [
    {
        "id": "extract_business_card",
        "category": "extraction",
        "difficulty": "easy",
        "prompt": "Extract the contact information from the following text into a JSON object with fields: name, phone, email, company, title.\n\nText: \"John Smith is a Senior Software Engineer at TechCorp Inc. You can reach him at john.smith@techcorp.com or call (555) 123-4567.\"",
        "expected_values": {
            "name": "John Smith",
            "phone": "(555) 123-4567",
            "email": "john.smith@techcorp.com",
            "company": "TechCorp Inc",
            "title": "Senior Software Engineer"
        },
        "schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string"},
                "phone": {"type": "string"},
                "email": {"type": "string"},
                "company": {"type": "string"},
                "title": {"type": "string"}
            },
            "required": ["name", "phone", "email", "company", "title"],
            "additionalProperties": False
        }
    },
    {
        "id": "extract_receipt",
        "category": "extraction",
        "difficulty": "medium",
        "prompt": "Extract receipt information from the following text into a JSON object with: store_name, date, items (array of objects with name and price), subtotal, tax, total.\n\nText: \"WALMART SUPERCENTER\nDate: 03/15/2026\nMilk 2% 1gal ........... $4.98\nSourdough Bread ........ $3.49\nOrganic Eggs 12ct ...... $5.99\nAvocados 3ct ........... $4.47\nSubtotal: $18.93\nTax (8.25%): $1.56\nTOTAL: $20.49\"",
        "expected_values": {
            "store_name": "WALMART SUPERCENTER",
            "date": "03/15/2026",
            "items": [
                {"name": "Milk 2% 1gal", "price": 4.98},
                {"name": "Sourdough Bread", "price": 3.49},
                {"name": "Organic Eggs 12ct", "price": 5.99},
                {"name": "Avocados 3ct", "price": 4.47}
            ],
            "subtotal": 18.93,
            "tax": 1.56,
            "total": 20.49
        },
        "schema": {
            "type": "object",
            "properties": {
                "store_name": {"type": "string"},
                "date": {"type": "string"},
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "price": {"type": "number"}
                        },
                        "required": ["name", "price"],
                        "additionalProperties": False
                    }
                },
                "subtotal": {"type": "number"},
                "tax": {"type": "number"},
                "total": {"type": "number"}
            },
            "required": ["store_name", "date", "items", "subtotal", "tax", "total"]
        }
    },
    {
        "id": "extract_api_log",
        "category": "extraction",
        "difficulty": "hard",
        "prompt": "Extract structured data from the following API log entry into JSON with: timestamp, method, path, status_code, response_time_ms, error (null if no error), and headers (object with content-type, x-request-id, user-agent).\n\nText: '[2026-05-05T10:23:45.678Z] POST /api/v2/users/authenticate -> 401 (145.3ms) | Headers: {\"content-type\": \"application/json\", \"x-request-id\": \"req-abc123def456\", \"user-agent\": \"MobileApp/3.2.1 (iOS 17.4)\"} | Error: Invalid credentials - email not verified'",
        "schema": {
            "type": "object",
            "properties": {
                "timestamp": {"type": "string"},
                "method": {"type": "string", "enum": ["GET", "POST", "PUT", "DELETE", "PATCH"]},
                "path": {"type": "string"},
                "status_code": {"type": "number"},
                "response_time_ms": {"type": "number"},
                "error": {"oneOf": [{"type": "string"}, {"type": "null"}]},
                "headers": {
                    "type": "object",
                    "properties": {
                        "content-type": {"type": "string"},
                        "x-request-id": {"type": "string"},
                        "user-agent": {"type": "string"}
                    },
                    "required": ["content-type", "x-request-id", "user-agent"],
                    "additionalProperties": False
                }
            },
            "required": ["timestamp", "method", "path", "status_code", "response_time_ms", "error", "headers"]
        }
    }
]

In [ ]:
import jsonschema

def extract_json(raw):
    """Try multiple strategies to extract JSON from model output."""
    text = raw.strip()
    # direct parse
    try:
        return json.loads(text)
    except:
        pass

    # strip markdown fences
    match = re.search(r'```(?:json)?\s*(.*?)\s*```', text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(1))
        except:
            pass

    # find JSON boundaries
    for start, end in [('{', '}'), ('[', ']')]:
        s = text.find(start)
        e = text.rfind(end)
        if s != -1 and e > s:
            try:
                return json.loads(text[s:e+1])
            except:
                pass
    return None

In [ ]:
def generate(prompt: str, system_prompt: str = SYSTEM_PROMPT) -> tuple:
    """Run a single generation using the Mistral tokenizer + model.
    Returns (response_text, latency_ms, num_tokens)."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": prompt},
    ]
    tokenized = tokenizer.encode_chat_completion(ChatCompletionRequest(messages=messages))
    input_ids = torch.tensor([tokenized.tokens])
    attention_mask = torch.ones_like(input_ids)

    start = time.time()
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=1024,
        temperature=0.15,
        do_sample=True,
    )
    elapsed = (time.time() - start) * 1000

    response = tokenizer.decode(outputs[0][len(tokenized.tokens):])
    num_tokens = outputs.shape[1] - input_ids.shape[1]
    return response, elapsed, num_tokens

In [ ]:
BENCHMARK_SYSTEM_PROMPT = "You are helpful assistant. Respond with ONLY valid JSON. No markdown, no code blocks, no explanation."

results = []

all_categories = [
    ("Simple JSON Tasks",   SIMPLE_JSON_TASKS),
    ("Schema Adherence",   SCHEMA_ADHERENCE_TASKS),
    ("Function Calling",   FUNCTION_CALLING_TASKS),
    ("Extraction",         EXTRACTION_TASKS),
]

for category_name, tasks in all_categories:
    print(f"\n{'='*60}")
    print(f"CATEGORY: {category_name}")
    print(f"{'='*60}")

    for task in tasks:
        response, elapsed, num_tokens = generate(task["prompt"], system_prompt=BENCHMARK_SYSTEM_PROMPT)
        tokens_per_sec = round(num_tokens / (elapsed / 1000), 1) if elapsed > 0 else 0

        # validation
        json_valid = False
        schema_valid = False
        parsed = None
        error_msg = None

        parsed = extract_json(response)
        if parsed is not None:
            json_valid = True
        else:
            error_msg = "Could not extract valid JSON from output"

        if json_valid:
            try:
                jsonschema.validate(instance=parsed, schema=task["schema"])
                schema_valid = True
            except jsonschema.ValidationError as e:
                schema_valid = False
                error_msg = f"Schema validation error: {e.message}"

        result = {
            "id": task["id"],
            "difficulty": task["difficulty"],
            "json_valid": json_valid,
            "schema_valid": schema_valid,
            "latency_ms": round(elapsed, 1),
            "tokens_generated": num_tokens,
            "tokens_per_sec": tokens_per_sec,
            "response": response,
            "error": error_msg,
        }
        results.append(result)

        print(f"\nResponse: {response[:200]}{'...' if len(response) > 200 else ''}")
        print(f"Latency: {round(elapsed, 1)} ms | Tokens: {num_tokens} | Speed: {tokens_per_sec} tok/s")
        print(f"Valid JSON: {'YES' if json_valid else 'NO'}")
        print(f"Schema Valid: {'YES' if schema_valid else 'NO'}")
        if error_msg:
            print(f"Error: {error_msg}")

In [ ]:
# --- Final Summary ---
print(f"\n\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")
print(f"{'Task ID':<25} {'Diff':<8} {'JSON':<8} {'Schema':<8} {'Latency':<10} {'tok/s':<8}")
print("-" * 70)

passed_json = 0
passed_schema = 0
for r in results:
    json_status = "Y" if r["json_valid"] else "N"
    schema_status = "Y" if r["schema_valid"] else "N"
    if r["json_valid"]:
        passed_json += 1
    if r["schema_valid"]:
        passed_schema += 1
    print(f"{r['id']:<25} {r['difficulty']:<8} {json_status:<8} {schema_status:<8} {r['latency_ms']:<10} {r['tokens_per_sec']:<8}")

print("-" * 70)
print(f"JSON Parse Rate:   {passed_json}/{len(results)} ({100*passed_json/len(results):.0f}%)")
print(f"Schema Valid Rate: {passed_schema}/{len(results)} ({100*passed_schema/len(results):.0f}%)")